# CreditMetrics Demo
This notebook demonstrates how to use the project after refactoring. The implementation lives in `src/`; this notebook simply orchestrates the workflow.

In [ ]:
from src.simulation import (
    generate_correlated_returns,
    map_returns_to_ratings,
    generate_recovery_rates,
)

from src.portfolio import (
    calculate_loan_forward_values,
    calculate_portfolio_values,
)

from src.risk_metrics import (
    calculate_losses,
    calculate_var,
    calculate_expected_shortfall,
)

from src.visualization import plot_loss_distribution


## 1. Define model parameters

In [ ]:
import numpy as np

In [2]:
NUM_SCENARIOS = 50000
CONFIDENCE_LEVELS = [0.95, 0.99]
np.random.seed(42)
 
# Ratings (ordered from best to worst)
RATINGS = ['Aa', 'Aa', 'A', 'Baa', 'Ba', 'B', 'Caa-C', 'Default']
RATING_INDEX = {rating: i for i, rating in enumerate(RATINGS)}

bond_names = ['Boyd Gaming Corp', 'Brinker International Inc', 'American Airlines Group Inc']

# Current exposures (in dollars)
exposures = {
    'Boyd Gaming Corp': 4_000_000,
    'Brinker International Inc': 5_000_000,
    'American Airlines Group Inc': 6_000_000
}

# Current forward values at t=0 (should be close to exposures) (Calculation can be found in data/valuation.csv)
current_values = {
    'Boyd Gaming Corp': 3_980_869,
    'Brinker International Inc': 5_125_176,
    'American Airlines Group Inc': 6_116_918
    }

initial_portfolio_value = sum(current_values.values())

# Forward values at 1-year horizon for each rating. (Calculation can be found in data/valuation.csv)
forward_values = {
    'Boyd Gaming Corp': {
        'Aaa': 4_345_450,
        'Aa': 4_317_394,
        'A': 4_305_235,
        'Baa': 4_282_173,
        'Ba': 4_218_112,
        'B': 4_130_678,
        'Caa-C': 3_928_288,
        'Default': 0  # Will be overridden by recovery × exposure
            },
    'Brinker International Inc': {
        'Aaa': 5_669_568,
        'Aa': 5_616_557,
        'A': 5_595_895,
        'Baa': 5_551_549,
        'Ba': 5_430_615,
        'B': 5_282_823,
        'Caa-C': 4_919_623,
        'Default': 0
    },
    'American Airlines Group Inc': {
        'Aaa': 7_178_778,
        'Aa': 7_092_932,
        'A': 7_062_203,
        'Baa': 6_989_991,
        'Ba': 6_793_624,
        'B': 6_578_192,
        'Caa-C': 6_014_339,
        'Default': 0
    }
}

# Cholesky matrix (3x3 lower triangular for 3 bonds) (Calculation can be found in data/correlation.csv)
cholesky_matrix = np.array([
    [1.0000, 0.0000, 0.0000],
    [0.5219, 0.8530, 0.0000],
    [0.4665, 0.2129, 0.8585]
])

# Rating thresholds.
# Structure: thresholds[bond][rating] = threshold value (Calculation can be found in data/thresholds.csv)
rating_thresholds = {
    "Boyd Gaming Corp": {
    "thresholds": np.array([-2.6336,-2.2361,-1.5621,2.6336]),
    "states": ["Default", "Caa-C", "B", "Ba", "Baa"]
    },
    "Brinker International Inc": {
    "thresholds": np.array([-2.6336,-2.2361,-1.5621,2.6336]),
    "states": ["Default", "Caa-C", "B", "Ba", "Baa"]
    },
    "American Airlines Group Inc": {
    "thresholds": np.array([-2.1082,0.8566,2.3759]),
    "states": ["Caa-C", "B", "Ba", "Baa"]
    }
}

# Recovery rate distribution parameters (Beta distribution). (Calculation can be found in data/valuation.csv)
# Same alpha and beta for all bonds (Senior Unsecured)
alpha = 8.3653      # Beta alpha parameter
beta_param = 13.7068 # Beta beta parameter

print(f"\nPortfolio Configuration:")
print(f"  Bonds: {', '.join(bond_names)}")
print(f"  Initial Portfolio Value: ${initial_portfolio_value:,.2f}")
print(f"  Number of Scenarios: {NUM_SCENARIOS:,}")
print(f"  Recovery Distribution: Beta(α={alpha}, β={beta_param})")
print(f"  Correlation Structure: Cholesky-induced")

NameError: name 'np' is not defined

## 2. Generate simulated scenarios

In [ ]:
print(f"Generating {NUM_SCENARIOS:,} correlated asset return scenarios...")
correlated_returns, uncorrelated_returns = generate_correlated_returns(
    NUM_SCENARIOS, cholesky_matrix, num_bonds=3
)

rint(f"\nMapping returns to rating scenarios using thresholds...")
rating_scenarios = map_returns_to_ratings(
    correlated_returns, rating_thresholds
)

print(f"  Uncorrelated returns shape: {uncorrelated_returns.shape}")
print(f"  Correlated returns shape: {correlated_returns.shape}")
print(f"  Correlation matrix (from data):\n{np.corrcoef(correlated_returns.T)}")

# Summary of rating distributions
for bond_name in bond_names:
    ratings = rating_scenarios[bond_name]
    unique, counts = np.unique(ratings, return_counts=True)
    print(f"\n  {bond_name}:")
    for rating_label, count in zip(unique, counts):
        pct = (count / NUM_SCENARIOS) * 100
        print(f"    {rating_label:8s}: {count:>6,} scenarios ({pct:>5.2f}%)")

print(f"\nGenerating uncorrelated recovery rates using Beta distribution...")
recovery_rates, uniform_random = generate_recovery_rates(
    NUM_SCENARIOS, alpha, beta_param, num_bonds=3
)

print(f"  Uniform random values shape: {uniform_random.shape}")
print(f"  Recovery rates shape: {recovery_rates.shape}")
print(f"  Recovery statistics:")
print(f"  Mean recovery: {np.mean(recovery_rates):.4f}")
print(f"  Std recovery:  {np.std(recovery_rates):.4f}")
print(f"  Min recovery:  {np.min(recovery_rates):.4f}")
print(f"  Max recovery:  {np.max(recovery_rates):.4f}")

## 3. Value the portfolio

In [ ]:
print(f"\nCalculating implied forward loan values...")
loan_forward_values = calculate_loan_forward_values(
    rating_scenarios, recovery_rates, forward_values, exposures
)

for bond_name in bond_names:
    values = loan_forward_values[bond_name]
    print(f"\n  {bond_name}:")
    print(f"    Mean forward value: ${np.mean(values):>15,.2f}")
    print(f"    Std forward value:  ${np.std(values):>15,.2f}")
    print(f"    Min forward value:  ${np.min(values):>15,.2f}")
    print(f"    Max forward value:  ${np.max(values):>15,.2f}")

print(f"\nAggregating to portfolio values...")
portfolio_values = calculate_portfolio_values(loan_forward_values)
    
print(f"  Portfolio values shape: {portfolio_values.shape}")
print(f"  Portfolio value statistics:")
print(f"  Mean portfolio value: ${np.mean(portfolio_values):>15,.2f}")
print(f"  Std portfolio value:  ${np.std(portfolio_values):>15,.2f}")
print(f"  Min portfolio value:  ${np.min(portfolio_values):>15,.2f}")
print(f"  Max portfolio value:  ${np.max(portfolio_values):>15,.2f}")


## 4. Calculate portfolio risk

In [ ]:
print(f"\nCalculating portfolio losses...")
losses = calculate_losses(portfolio_values, initial_portfolio_value)
    
print(f"  Loss statistics:")
print(f"    Mean loss:    ${np.mean(losses):>15,.2f}")
print(f"    Median loss:  ${np.median(losses):>15,.2f}")
print(f"    Std loss:     ${np.std(losses):>15,.2f}")
print(f"    Min loss:     ${np.min(losses):>15,.2f}")
print(f"    Max loss:     ${np.max(losses):>15,.2f}")

print(f"\nCalculating Absolute VaR and Expected Shortfall...")
results = calculate_var_and_es(losses, CONFIDENCE_LEVELS)

for conf in sorted(CONFIDENCE_LEVELS):
    conf_pct = int(conf * 100)
    var = results[conf]['VaR']
    es = results[conf]['ES']
        
print(f"\nConfidence Level: {conf_pct}%")
print(f"  Absolute VaR:              ${var:>15,.2f}")
print(f"  Absolute Expected Shortfall: ${es:>15,.2f}")
print(f"  Tail Observations:         {results[conf]['num_tail']:>15,} ({results[conf]['pct_tail']:.2f}%)")


## 5. Visualise results

In [ ]:
plot_loss_distribution(losses, results)

## What moved into `src/`

- `simulation.py` – Monte Carlo engine, correlated returns, rating migrations, recovery generation.
- `portfolio.py` – instrument and portfolio valuation.
- `risk_metrics.py` – losses, VaR, Expected Shortfall.
- `visualization.py` – plotting functions.

This notebook should remain short and readable; the heavy lifting happens in the modules.
